Feature Engineering and Feature Selection

Простая модель обученная на хороших данных, будет лучше, чем сложный ансамбль на грязных данных.

Три похожые, но разные задачи:
- feature extraction && feature engineering: преобразование данных в функции пригодные для моделирования;
- feature transformation: трансформирование данных, чтобы улучшить алгоритм;
- feature selection: удаление необязательных фич;

#Feature Extraction
#Тексты
Перед тем как работать с тестами, мы должны токенизировать их. Токенизация - разбиение тектса на юниты (иногда юнит = одно слово). После токенизации, данные нужно нормализовать. Мы представили документ как последовательность слов. Теперь нужно превратить их в векторы.

###Bag of Words
Cоздаем вектор, размерность = кол-ву уникальных юнитов, и считаем кол-во включения каждого слова в тексте.

In [9]:
import numpy as np

texts = ["i have a cat", "you have a dog", "you and i have a cat and a dog"]

vocabulary = list(
    enumerate(set([word for sentence in texts for word in sentence.split()]))
)

print(f"Vocabulary: {vocabulary}")

def vectorize(text):
    vector = np.zeros(len(vocabulary))
    for i, word in vocabulary:
        num = 0
        for w in text:
            if w == word:
                num += 1
        if num:
            vector[i] = num
    return vector

print("Vectors:")
for sentence in texts:
    print(vectorize(sentence.split()))

Vocabulary: [(0, 'i'), (1, 'cat'), (2, 'you'), (3, 'dog'), (4, 'have'), (5, 'a'), (6, 'and')]
Vectors:
[1. 1. 0. 0. 1. 1. 0.]
[0. 0. 1. 1. 1. 1. 0.]
[1. 1. 1. 1. 1. 2. 2.]


Это слишком наивный подход. На практике нужны стоп слова, максимальный размер словаря, и более эффективные структуры данных.

Используя этот алгоритм, мы теряем последовательность слов в тексте. Чтобы этого избежать мы можем использовать n-граммы при токенизации.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(ngram_range=(1, 1))
print(
    vect.fit_transform(["no i have cows", "i have no cows"]).toarray(),
    vect.vocabulary_
)

[[1 1 1]
 [1 1 1]] {'no': 2, 'have': 1, 'cows': 0}


In [15]:
vect = CountVectorizer(ngram_range=(1, 2))
print(
    vect.fit_transform(["no i have cows", "i have no cows"]).toarray(),
    vect.vocabulary_
)

[[1 1 1 0 1 0 1]
 [1 1 0 1 1 1 0]] {'no': 4, 'have': 1, 'cows': 0, 'no have': 6, 'have cows': 2, 'have no': 3, 'no cows': 5}


Не обязательно использовать только слова. В некоторых случаях можно генерировать N-граммы символов. Такой подход позволит учесть сходство связанных слов или устранить опечатки.

In [20]:
from scipy.spatial.distance import euclidean
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(ngram_range=(3, 3), analyzer="char_wb")

n1, n2, n3, n4 = vect.fit_transform(
    ["andersen", "petersen", "petrov", "smith"]
).toarray()

euclidean(n1, n2), euclidean(n2, n3), euclidean(n3, n4)

(np.float64(2.8284271247461903),
 np.float64(3.1622776601683795),
 np.float64(3.3166247903554))

Дополняя идею "мешка слов": слова, которые редко встречаются в корпусе (во всех документах датасета), но присутствуют в конкретном документе, могут быть более важными. Поэтому логично увеличить вес более специфичных для данной области слов, чтобы отличить их от общеупотребительных.

Такой подход называется **TF-IDF** (частота термина — обратная частота документа).

\begin{align*}
idf(t, D) &= \log \frac{|D|}{df(d, t) + 1} \\
tfidf(t, d, D) &= tf(t, d) \times idf(t, D)
\end{align*}

Пояснение:
- $tf(t, d)$ — Term Frequency: сколько раз термин t встречается в документе d.
- $df(d, t)$ — Document Frequency: в скольких документах коллекции D встречается термин t.
- $|D|$ — общее количество документов в коллекции.
- $idf(t, D)$ — Inverse Document Frequency: отражает, насколько редко слово встречается в коллекции.
- $tfidf(t, d, D)$ — итоговое значение, показывающее важность термина t в документе d с учётом всей коллекции D.


Используя эти алгортмы, мы можем получить простое решение проблемы, которе мы можем исспользовать как бейзлайн.

##Word2Vec
Word2Vec — это частный случай алгоритмов представления слов в виде эмбеддингов. Используя Word2Vec и похожие модели, мы можем не только векторизовать слова в пространстве высокой размерности (обычно несколько сотен измерений), но и сравнивать их семантическое сходство. Классический пример операций над векторизованными понятиями:
king - man + woman = queen.

Для того чтобы координаты векторов действительно отражали смысл слов, такие модели необходимо обучать на очень больших датасетах.

Похожие методы применяются и в других областях, таких как биоинформатика. Неожиданное применение — food2vec (векторизация продуктов питания).

#Изображения

Работать с изображениями одновременно и проще, и сложнее. Это проще, потому что можно просто использовать одну из популярных предварительно подготовленных сетей, не задумываясь, но сложнее, потому что, если вам нужно вникнуть в детали, вы можете в конечном итоге углубиться в них.

До появления мощных GPU и "ренессанса нейросетей", извлечение признаков из изображений было сложной и ручной задачей: нужно было находить углы, границы, статистику цветов и т.д. Опытные специалисты по компьютерному зрению отмечали сходство старых методов с нейросетями — например, свёрточные слои напоминают каскады Хаара.

Сейчас для задач с изображениями чаще всего используют свёрточные нейросети (CNN). Не обязательно обучать их с нуля — можно взять предобученную сеть и сделать fine-tuning: заменить последние полносвязные слои на новые, подходящие под задачу, и дообучить сеть. Если нужно просто векторизовать изображение (например, для классификатора вне нейросети), достаточно удалить последние слои и использовать выход предыдущих.

![alt text](https://cdn-images-1.medium.com/max/800/1*Iw_cKFwLkTVO2SPrOZU2rQ.png)

#Геопространственные данные

Геоданные не так часто встречаются в задачах, но полезно освоить базовые методы работы с ними, особенно учитывая наличие множества готовых решений.

Обычно геоданные представлены в виде адресов или координат (широта, долгота). В зависимости от задачи, могут потребоваться:

Геокодирование — получение координат по адресу.
Обратное геокодирование — получение адреса по координатам.
Обе операции доступны через внешние API, например Google Maps или OpenStreetMap. Разные сервисы работают с разным качеством в зависимости от региона. Удобная библиотека geopy предоставляет обёртку над этими сервисами.

Если данных много, API быстро достигает лимитов, а HTTP-запросы не всегда быстрые. В таком случае стоит рассмотреть возможность локального использования OpenStreetMap.

Если данных немного, и не требуется извлекать сложные признаки, можно использовать библиотеку reverse_geocoder как простую альтернативу.

При работе с **геокодированием** важно учитывать:

- **Адреса** могут содержать опечатки — необходима предварительная **очистка данных**.
- **Координаты** ошибок не содержат, но могут быть неточными из-за:
  - GPS-шумов;
  - плохого сигнала (туннели, густая застройка и т.д.);
  - или если данные получены не по GPS, а по **Wi-Fi сетям** (особенно в мобильных устройствах).

Пример: при передвижении по Манхэттену можно внезапно получить координаты из Чикаго — так работает Wi-Fi геолокация.

###Почему возникают ошибки при Wi-Fi геолокации?

- Используется комбинация **SSID** и **MAC-адреса**;
- Некоторые провайдеры используют **одинаковое оборудование** с одинаковыми MAC-адресами в разных городах;
- Компании могут **переезжать с роутерами**, но координаты остаются прежними.


###Генерация признаков на основе инфраструктуры

Часто геоточка находится **внутри городской инфраструктуры**. Здесь можно применить **житейскую логику** и извлечь полезные признаки:

- Близость к станции метро;
- Количество этажей здания;
- Расстояние до ближайшего магазина;
- Количество банкоматов и т.п.

###Задачи за пределами города

- Вне городской среды полезны признаки из других источников:
  - **Высота над уровнем моря** и т.д.

###Признаки из маршрутов между точками

Если точки связаны маршрутом, полезны следующие признаки:

- Расстояние:
  - по прямой (**Great Circle Distance**);
  - по дорогам (по графу маршрутов);
- Количество поворотов;
- Соотношение левых и правых поворотов;
- Количество:
  - светофоров;
  - перекрёстков;
  - мостов и развязок.


#Дата и время

Начнем с самого простого — дня недели, который можно легко превратить в 7 дамми-переменных с помощью one-hot кодирования. Также создаем бинарный признак is_weekend, указывающий, является ли день выходным:

```python
df['dow'] = df['created'].apply(lambda x: x.date().weekday())
df['is_weekend'] = df['created'].apply(lambda x: 1 if x.date().weekday() in (5, 6) else 0)
```
💡 Но этого может быть недостаточно. Некоторые задачи требуют дополнительных календарных признаков:

Снятие наличных может быть связано с днем зарплаты. Покупка проездного — с началом месяца. Аномалии — с праздниками, погодными условиями, событиями.

Работа с временными данными (часы, минуты, дни месяца и т. д.) не так проста, как может показаться. Если использовать время как реальную переменную, возникает некоторое противоречие с природой данных: $0 \leq \text{часы} \leq 23$, однако временной интервал $00:00:00, 02.01$ больше, чем $01.01, 23:00:00$. Для некоторых задач это может быть критично. В то же время, если закодировать время как категориальные переменные, это приведет к избыточному числу признаков и потере информации о близости: разница между 22 и 23 часами будет такой же, как разница между 22 и 7.

Существуют и более эзотерические подходы к обработке таких данных, например, проекция времени на окружность и использование двух координат.

Эта трансформация сохраняет расстояние между точками, что важно для алгоритмов, оценивающих расстояние (kNN, SVM, k-means и др.).

In [ ]:
def make_harmonic_features(value, period=24):
    value *= 2 * np.pi / period
    return np.cos(value), np.sin(value)

# Временные ряды, web и т. д.

Если вы работаете с веб-данными, то обычно у вас есть информация о User Agent пользователя. Это кладезь информации. Во-первых, нужно извлечь операционную систему из него. Во-вторых, создать признак is_mobile. В-третьих, посмотреть на браузер.

In [22]:
import user_agents

ua = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Ubuntu Chromium/56.0.2924.76 Chrome/56.0.2924.76 Safari/537.36"
ua = user_agents.parse(ua)

print("Is a bot? ", ua.is_bot)
print("Is mobile? ", ua.is_mobile)
print("Is PC? ", ua.is_pc)
print("OS Family: ", ua.os.family)
print("OS Version: ", ua.os.version)
print("Browser Family: ", ua.browser.family)
print("Browser Version: ", ua.browser.version)

Is a bot?  False
Is mobile?  False
Is PC?  True
OS Family:  Ubuntu
OS Version:  ()
Browser Family:  Chromium
Browser Version:  (56, 0, 2924)


Как и в других областях, можно придумать собственные признаки, основываясь на интуиции о природе данных. 

Следующая полезная информация — это IP-адрес, из которого можно извлечь страну и, возможно, город, провайдера и тип подключения (мобильное/стационарное). Нужно понимать, что существует множество прокси-серверов и устаревших баз данных, поэтому эта информация может содержать шум. Гуру сетевого администрирования могут пытаться извлечь еще более изысканные признаки, например, предложения по использованию VPN. Кстати, данные с IP-адреса хорошо комбинируются с http_accept_language: если пользователь сидит за чилийским прокси, а локаль браузера ru_RU, это что-то подозрительное, и стоит обратить внимание на соответствующий столбец в таблице (например, is_traveler_or_proxy_user).

Feature transformations

##Нормализация и изменение распределения

Простой пример: предположим, что задача состоит в том, чтобы предсказать стоимость квартиры на основе двух переменных — расстояния от центра города и количества комнат. Количество комнат редко превышает 5, в то время как расстояние от центра города может легко исчисляться тысячами метров.

`StandartScaler`: $\Large z = \frac{x - \mu}{\sigma}$

`MinMaxScaler`: $\Large X_{norm} = \frac{X - X_{min}}{X_{max}-X_{min}}$

##Взаимодействие
Если предыдущие преобразования казались больше математически обоснованными, то эта часть ближе к интуитивному анализу природы данных. Это можно отнести как к трансформации признаков, так и к генерации новых признаков.

```python
rooms = df["bedrooms"].apply(lambda x: max(x, 0.5))
Avoid division by zero; .5 is chosen more or less arbitrarily
df["price_per_bedroom"] = df["price"] / rooms
```

При генерации новых признаков важно не переусердствовать:

- Если количество исходных признаков ограничено, можно попробовать сгенерировать все возможные взаимодействия между ними.
- Однако затем необходимо отфильтровать лишние признаки — для этого используются методы отбора признаков, которые будут описаны в следующем разделе.
- Не все взаимодействия между признаками имеют физический или логический смысл.

**Пример:** полиномиальные признаки (PolynomialFeatures из sklearn.preprocessing) часто применяются в линейных моделях, но при этом почти не интерпретируемы.

Feature selection

На первый взгляд, идея отбора признаков может показаться неинтуитивной — ведь кажется, что больше данных всегда лучше. Однако существует как минимум две веские причины, по которым стоит избавляться от неинформативных признаков.

1) Снижение вычислительной сложности.
2) Борьба с переобучением.

#Статистический подход
Наиболее очевидным кандидатом на удаление является признак, значение которого остается неизменным, т.е. он вообще не содержит никакой информации. Если исходить из этой мысли, то разумно сказать, что признаки с низкой дисперсией хуже, чем с высокой. 

In [30]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import VarianceThreshold

x_data_generated, y_data_generated = make_classification()
x_data_generated.shape

(100, 20)

In [31]:
VarianceThreshold(0.9).fit_transform(x_data_generated).shape

(100, 16)

##Selection by modeling

In [ ]:
Synthetic example

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

x_data_generated, y_data_generated = make_classification(random_state=17)

logit = LogisticRegression(solver="lbfgs", random_state=17)
rf = RandomForestClassifier(n_estimators=10, random_state=17)
pipe = make_pipeline(SelectFromModel(estimator=rf), logit)

print(
    cross_val_score(
        logit, x_data_generated, y_data_generated, scoring="neg_log_loss", cv=5
    ).mean()
)
print(
    cross_val_score(
        rf, x_data_generated, y_data_generated, scoring="neg_log_loss", cv=5
    ).mean()
)
print(
    cross_val_score(
        pipe, x_data_generated, y_data_generated, scoring="neg_log_loss", cv=5
    ).mean()
)

-0.14386531116258758
-0.19042398084844722
-0.1265539336369222


Но стотит понимать, что это может ухудшить производительность

In [ ]:
x_data, y_data = get_data()
x_data = x_data_generated
y_data = y_data_generated

pipe1 = make_pipeline(StandardScaler(), SelectFromModel(estimator=rf), logit)

pipe2 = make_pipeline(StandardScaler(), logit)

print(
    "LR + selection: ",
    cross_val_score(pipe1, x_data, y_data, scoring="neg_log_loss", cv=5).mean(),
)
print(
    "LR: ", cross_val_score(pipe2, x_data, y_data, scoring="neg_log_loss", cv=5).mean()
)
print("RF: ", cross_val_score(rf, x_data, y_data, scoring="neg_log_loss", cv=5).mean())

LR + selection:  -0.13493587437744825
LR:  -0.1564046676197639
RF:  -0.19042398084844722


## Grid search
Наиболее надёжным, но при этом и самым ресурсоёмким методом отбора признаков является полный перебор (grid search). Методика заключается в следующем: обучается модель на различных подмножествах признаков, сохраняются метрики качества, после чего сравниваются результаты и выбирается наилучшее множество признаков. Такой подход называется Exhaustive Feature Selection.

Полный перебор всех возможных комбинаций признаков может быть чрезмерно затратным по времени, особенно при большом количестве признаков. Для снижения сложности используется ограничение пространства поиска. Один из вариантов — Sequential Feature Selection:

- Фиксируется небольшое число признаков N.

- Перебираются все комбинации N признаков.

- Выбирается комбинация с наилучшей метрикой.

- К ней последовательно добавляются новые признаки по одному.
  
- Алгоритм продолжается до достижения:
  - заданного максимального числа признаков;
  - или момента, когда качество модели перестаёт расти.

Существует обратный вариант алгоритма: он начинается с полного множества признаков, и признаки удаляются по одному до тех пор, пока это не начинает ухудшать качество модели или пока не достигнуто заданное количество признаков.

In [ ]:
# Install mlxtend
from mlxtend.feature_selection import SequentialFeatureSelector

selector = SequentialFeatureSelector(
    logit, scoring="neg_log_loss", verbose=2, k_features=3, forward=False, n_jobs=-1
)

selector.fit(x_data, y_data)